# 📊 Multi-Model Independent Evaluator & Leaderboard

This interactive dashboard allows you to define multiple YOLO models (standard or SAHI) and run an independent evaluation across a shared Test Dataset. Features:
- **Interactive Configuration**: Add dynamically N models, their paths, and SAHI configurations.
- **Quantitative Leaderboard**: Calculate metrics (Precision, Recall, F1-Score, mAP50) natively and sort the results.
- **Visual Comparison**: Upload a 4K raw image and view `1xN` subplots displaying their predictive differences side-by-side.

In [1]:
# Install libraries if not available
!pip install -q ultralytics ipywidgets shapely opencv-python matplotlib pandas sahi

import os
import sys
from pathlib import Path
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib import pyplot as plt
import pandas as pd
from ultralytics import YOLO

# Add root directory to python path to import our custom pipeline
project_root = str(Path(os.path.abspath('')).parent.parent)
if project_root not in sys.path:
    sys.path.append(project_root)

# Import unconditionally so ImportErrors bubble up and restart kernel is obvious
from src.inference import RockSegmentationPipeline, RockVisualizer

print("✅ Libraries loaded successfully! (Make sure to restart kernel if you edited src/inference.py recently)")

✅ Libraries loaded successfully! (Make sure to restart kernel if you edited src/inference.py recently)


## ⚙️ 1. Interactive Model Configuration

Define your test dataset `data.yaml` path, choose the number of models to compare, and input their parameters.
Check the **"Use SAHI"** box if the respective model should use slicing inference.

In [ ]:
# -------------------- UI Configuration --------------------
style = {'description_width': 'initial'}
layout = widgets.Layout(width='500px')

# Global storage mapping
gui_config = {
    "dataset_yaml": "",
    "models": []
}

default_yaml = str(Path.home() / "projects/DwiAnggara/Datasets/Batch3and4_YOLO/data.yaml")
if not os.path.exists(default_yaml):
    default_yaml = ""

dataset_widget = widgets.Text(
    value=default_yaml,
    description='Dataset YAML Path:',
    style=style, layout=widgets.Layout(width='700px')
)

num_models_widget = widgets.BoundedIntText(
    value=2, min=1, max=10, step=1,
    description='Number of Models:',
    style=style, layout=widgets.Layout(width='200px')
)

generate_btn = widgets.Button(description='Generate Inputs', button_style='info')
save_config_btn = widgets.Button(description='Save Config', button_style='success', icon='save')
config_output = widgets.Output()

models_container = widgets.VBox([])
model_widgets_mapping = []

def generate_model_inputs(b):
    models_container.children = []
    model_widgets_mapping.clear()
    
    rows = []
    for i in range(num_models_widget.value):
        row_title = widgets.HTML(f"<b>Model {i+1}</b>")
        name_input = widgets.Text(value=f"YOLO_M{i+1}", description="Name:", layout=widgets.Layout(width='200px'))
        path_input = widgets.Text(value="", placeholder="Paste best.pt absolute path...", description="Path:", layout=widgets.Layout(width='450px'))
        sahi_check = widgets.Checkbox(value=False, description="Use SAHI", layout=widgets.Layout(width='150px'))
        
        row_box = widgets.HBox([name_input, path_input, sahi_check])
        rows.append(widgets.VBox([row_title, row_box]))
        
        model_widgets_mapping.append({
            "name": name_input,
            "path": path_input,
            "use_sahi": sahi_check
        })
        
    models_container.children = rows

def save_configuration(b):
    with config_output:
        clear_output()
        dataset_yaml = dataset_widget.value
        
        if not os.path.exists(dataset_yaml):
            print(f"❌ Error: Dataset file not found at {dataset_yaml}")
            return
            
        gui_config['dataset_yaml'] = dataset_yaml
        gui_config['models'] = []
        
        valid = True
        for idx, w in enumerate(model_widgets_mapping):
            m_name = w['name'].value
            m_path = w['path'].value
            m_sahi = w['use_sahi'].value
            
            if not os.path.exists(m_path):
                print(f"❌ Error: Weights for '{m_name}' not found at {m_path}")
                valid = False
            else:
                gui_config['models'].append({
                    "name": m_name,
                    "path": m_path,
                    "use_sahi": m_sahi
                })
        
        if valid:
            print("✅ Configuration saved successfully!")
            print(f"📂 Dataset: {dataset_yaml}")
            for m in gui_config['models']:
                sahi_flag = "SAHI" if m['use_sahi'] else "Standard"
                print(f"  🤖 [{sahi_flag}] {m['name']} -> {os.path.basename(m['path'])}")

generate_btn.on_click(generate_model_inputs)
save_config_btn.on_click(save_configuration)

# Initialize defaults
generate_model_inputs(None)

display(widgets.VBox([
    dataset_widget,
    widgets.HBox([num_models_widget, generate_btn]),
    widgets.HTML("<hr>"),
    models_container,
    widgets.HTML("<br>"),
    save_config_btn,
    config_output
]))

## 📈 2. Quantitative Leaderboard Generation

Once your configuration is saved above, run the cell below to extract native evaluation metrics for each model against the test dataset. The loop collects Precision, Recall, Validation mAP, masks IoU, and F1-Scores to rank them. SAHI quantitative logic defaults to native sliced metrics on pre-sliced `test` data.

In [3]:
import logging
import cv2
import yaml
import json
import time
import glob
import numpy as np
from pathlib import Path
from IPython.display import HTML
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
except ImportError:
    print("⚠️ pycocotools not found! Installing now...")
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "pycocotools"])
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

# Suppress Ultralytics verbose validation logs for clean UI out
logging.getLogger("ultralytics").setLevel(logging.ERROR)

evaluate_btn = widgets.Button(description='Run Benchmark Evaluator', button_style='primary', icon='rocket')
eval_output = widgets.Output()

def extract_metrics(results_obj):
    if hasattr(results_obj, 'seg') and results_obj.seg is not None:
        p = float(results_obj.seg.mp)
        r = float(results_obj.seg.mr)
        m_50 = float(results_obj.seg.map50)
        m_all = float(results_obj.seg.map)
    elif hasattr(results_obj, 'box'):
        p = float(results_obj.box.mp)
        r = float(results_obj.box.mr)
        m_50 = float(results_obj.box.map50)
        m_all = float(results_obj.box.map)
    else:
        p, r, m_50, m_all = 0.0, 0.0, 0.0, 0.0
    return p, r, m_50, m_all

def compute_f1(p, r):
    return (2 * p * r / (p + r)) if (p + r) > 0 else 0.0

def build_coco_ground_truth(dataset_yaml_path):
    """Converts YOLO test split to COCO JSON format in memory."""
    with open(dataset_yaml_path, 'r') as f:
        data_info = yaml.safe_load(f)
        
    dataset_dir = Path(dataset_yaml_path).parent
    test_img_dir = dataset_dir / data_info.get('test', 'test/images')
    if 'images' not in str(test_img_dir):
        test_img_dir = Path(str(test_img_dir).replace('test', 'test/images'))
    test_lbl_dir = dataset_dir / str(data_info.get('test', 'test/images')).replace('images', 'labels')
    
    names = data_info.get('names', {})
    if isinstance(names, list):
        names = {i: n for i, n in enumerate(names)}
        
    coco_gt = {
        "images": [],
        "annotations": [],
        "categories": [{"id": k, "name": v} for k, v in names.items()]
    }
    
    img_files = glob.glob(str(test_img_dir / "*.jpg")) + glob.glob(str(test_img_dir / "*.png"))
    ann_id = 1
    
    for img_id, img_path in enumerate(img_files):
        img_name = Path(img_path).name
        img_bgr = cv2.imread(img_path)
        if img_bgr is None: continue
        h, w = img_bgr.shape[:2]
        
        coco_gt["images"].append({"id": img_id, "file_name": img_name, "width": w, "height": h})
        
        lbl_path = test_lbl_dir / (Path(img_path).stem + ".txt")
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5: # polygon
                    cls_id = int(parts[0])
                    coords = np.array([float(x) for x in parts[1:]]).reshape(-1, 2)
                    coords[:, 0] *= w
                    coords[:, 1] *= h
                    poly = coords.astype(np.float32)
                    
                    x, y, bw, bh = cv2.boundingRect(poly.astype(np.int32))
                    area = cv2.contourArea(poly)
                    if area > 0:
                        coco_gt["annotations"].append({
                            "id": ann_id,
                            "image_id": img_id,
                            "category_id": cls_id,
                            "segmentation": [poly.ravel().tolist()],
                            "area": area,
                            "bbox": [float(x), float(y), float(bw), float(bh)],
                            "iscrowd": 0
                        })
                        ann_id += 1
                        
    return coco_gt, img_files

def evaluate_custom_pipeline(model_path, dataset_yaml_path, is_sahi=False):
    """Runs the custom pipeline over the test dataset and evaluates using PyCOCOtools."""
    coco_gt_dict, img_files = build_coco_ground_truth(dataset_yaml_path)
    
    gt_path = "temp_gt.json"
    with open(gt_path, 'w') as f:
        json.dump(coco_gt_dict, f)
        
    cocoGt = COCO(gt_path)
    
    model = YOLO(model_path)
    post_processor = RockSegmentationPipeline(solidity_thr=0.85, min_area=300)
    
    coco_preds = []
    ann_id = 1
    
    start_time = time.time()
    for img_info in coco_gt_dict["images"]:
        img_id = img_info["id"]
        img_path = str(Path(dataset_yaml_path).parent / "test/images" / img_info["file_name"])
        img_bgr = cv2.imread(img_path)
        if img_bgr is None: continue
        
        # Base Prediction (No TTA)
        res = model.predict(img_bgr, conf=0.25, augment=False, verbose=False)[0]
        predictions = post_processor.run_pipeline(res)
        
        for pred in predictions:
            poly = pred["polygon"].astype(np.float32)
            cls_id = pred["class_id"]
            score = pred["score"]
            x, y, bw, bh = cv2.boundingRect(poly.astype(np.int32))
            area = cv2.contourArea(poly)
            if area > 0:
                coco_preds.append({
                    "image_id": img_id,
                    "category_id": int(cls_id),
                    "segmentation": [poly.ravel().tolist()],
                    "bbox": [float(x), float(y), float(bw), float(bh)],
                    "score": float(score),
                    "area": float(area)
                })
                    
    total_latency = time.time() - start_time
    avg_latency = (total_latency / len(img_files)) * 1000 if img_files else 0
    
    pred_path = "temp_pred.json"
    with open(pred_path, 'w') as f:
        json.dump(coco_preds, f)
        
    p, r, m_50, m_all = 0.0, 0.0, 0.0, 0.0
    if len(coco_preds) > 0:
        cocoDt = cocoGt.loadRes(pred_path)
        cocoEval = COCOeval(cocoGt, cocoDt, 'segm')
        cocoEval.evaluate()
        cocoEval.accumulate()
        cocoEval.summarize()
        
        # Pulling standard COCO metrics
        m_all = float(cocoEval.stats[0]) # mAP @ [IoU=0.50:0.95]
        m_50 = float(cocoEval.stats[1])  # mAP @ IoU=0.50
        r = float(cocoEval.stats[8])     # AR @ IoU=0.50:0.95 maxDets=100
        
        precisions = cocoEval.eval['precision']
        p = float(np.mean(precisions[0, :, :, 0, -1][precisions[0, :, :, 0, -1] > -1]))
        if np.isnan(p): p = 0.0
        
    return p, r, m_50, m_all, avg_latency

def run_benchmarks(b):
    with eval_output:
        clear_output(wait=True)
        if not gui_config['models']:
            print("⚠️ Please configure and save your models first.")
            return
            
        dataset_yaml = gui_config['dataset_yaml']
        base_results = []
        pp_results = []
        
        for cfg in gui_config['models']:
            print(f"\n⏳ Starting evaluation for: {cfg['name']} {'(SAHI Mode)' if cfg['use_sahi'] else '(Standard Mode)'}")
            
            try:
                model = YOLO(cfg['path'])
                
                # --- Base Validation ---
                print(f"   🚀 Validating natively against test split (Base)...")
                res_base = model.val(data=dataset_yaml, split='test', augment=False, conf=0.25, verbose=False)
                p, r, map50, map50_95 = extract_metrics(res_base)
                f1_score = compute_f1(p, r)
                
                latency = sum(res_base.speed.values()) if hasattr(res_base, 'speed') else 0.0
                fps = 1000.0 / latency if latency > 0 else 0.0
                
                base_results.append({
                    "Model Name": cfg['name'],
                    "Type": "SAHI" if cfg['use_sahi'] else "Standard",
                    "F1-Score": f"{f1_score * 100:.2f}%",
                    "Precision": f"{p * 100:.2f}%",
                    "Recall": f"{r * 100:.2f}%",
                    "mAP@50": f"{map50 * 100:.2f}%",
                    "mAP@50-95": f"{map50_95 * 100:.2f}%",
                    "Avg Latency": f"{latency:.1f}ms",
                    "FPS": f"{fps:.1f}",
                    "_f1_raw": f1_score
                })
                
                # --- Watershed Validation ---
                print(f"   🔮 Validating with Custom PyCOCO Evaluator (Morphological Post-Processor)...")
                tp, tr, tmap50, tmap50_95, pp_latency = evaluate_custom_pipeline(cfg['path'], dataset_yaml, cfg['use_sahi'])
                tf1_score = compute_f1(tp, tr)
                pp_fps = 1000.0 / pp_latency if pp_latency > 0 else 0.0
                
                pp_results.append({
                    "Model Name": cfg['name'],
                    "Type": "SAHI + PostProcess" if cfg['use_sahi'] else "Standard + PostProcess",
                    "F1-Score": f"{tf1_score * 100:.2f}%",
                    "Precision": f"{tp * 100:.2f}%",
                    "Recall": f"{tr * 100:.2f}%",
                    "mAP@50": f"{tmap50 * 100:.2f}%",
                    "mAP@50-95": f"{tmap50_95 * 100:.2f}%",
                    "Avg Latency": f"{pp_latency:.1f}ms",
                    "FPS": f"{pp_fps:.1f}",
                    "_f1_raw": tf1_score
                })
                print(f"   ✅ Done.")
                
            except Exception as e:
                print(f"   ❌ Error evaluating {cfg['name']}: {e}")
                import traceback
                traceback.print_exc()
                
        if base_results:
            df_base = pd.DataFrame(base_results).sort_values(by="_f1_raw", ascending=False).reset_index(drop=True)
            df_base.insert(0, "Rank", df_base.index + 1)
            df_base = df_base.drop(columns=["_f1_raw"])
            
            df_pp = pd.DataFrame(pp_results).sort_values(by="_f1_raw", ascending=False).reset_index(drop=True)
            df_pp.insert(0, "Rank", df_pp.index + 1)
            df_pp = df_pp.drop(columns=["_f1_raw"])
            
            display(HTML("<h3>🛑 TABLE 1: Quantitative Leaderboard (Raw Native YOLO Base)</h3>"))
            display(df_base)
            
            display(HTML("<h3>🪄 TABLE 2: Quantitative Leaderboard (Post-Processed with Watershed)</h3>"))
            display(df_pp)

evaluate_btn.on_click(run_benchmarks)
display(evaluate_btn, eval_output)

Button(button_style='primary', description='Run Benchmark Evaluator', icon='rocket', style=ButtonStyle())

Output()

## 👁️ 3. Visual Multi-Model Comparison

Upload a raw 4K image to observe the prediction differences among all configured models visually side-by-side. The hollow polygon output mirrors CVAT annotation styles and utilizes exact coordinate projections.

In [4]:
upload_compare_btn = widgets.FileUpload(accept='image/*', multiple=False, description='Upload Test Image', style=style, layout=widgets.Layout(width='300px'))
conf_slider = widgets.FloatSlider(value=0.25, min=0.05, max=0.95, step=0.05, description='Confidence:', style=style)
min_area_slider = widgets.IntSlider(value=300, min=0, max=5000, step=50, description='Min Area (px²):', style=style)
solidity_slider = widgets.FloatSlider(value=0.85, min=0.50, max=0.99, step=0.01, readout_format='.2f', description='Solidity Thr:', style=style)

visualize_btn = widgets.Button(description='Compare visually', button_style='warning', icon='eye')
compare_output = widgets.Output()

def run_multi_visualizer(b):
    with compare_output:
        clear_output(wait=True)
        if not gui_config['models']:
            print("⚠️ Please configure and save your models first.")
            return
            
        if not upload_compare_btn.value:
            print("⚠️ Please upload an image first.")
            return
            
        print("⏳ Processing visual comparisons across all models...")
        
        # Read image
        uploaded_file = list(upload_compare_btn.value.values())[0] if isinstance(upload_compare_btn.value, dict) else upload_compare_btn.value[0]
        content = uploaded_file['content']
        nparr = np.frombuffer(content, np.uint8)
        original_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        num_models = len(gui_config['models'])
        
        # Pipeline Instantiation for morphological splitting only
        post_processor = RockSegmentationPipeline(solidity_thr=solidity_slider.value, min_area=min_area_slider.value)
        visualizer = RockVisualizer(thickness=8)
        
        for cfg in gui_config['models']:
            raw_polygons_by_class = {}
            proc_polygons_by_class = {}
            print(f"\n   -> Processing {cfg['name']} (Standard + Watershed)...")
            
            try:
                if cfg['use_sahi']:
                    from sahi import AutoDetectionModel
                    from sahi.predict import get_sliced_prediction
                    import torch
                    
                    device_mode = "cuda:0" if torch.cuda.is_available() else "cpu"
                    
                    sahi_model = AutoDetectionModel.from_pretrained(
                        model_type="yolov8",
                        model_path=cfg['path'],
                        confidence_threshold=conf_slider.value,
                        device=device_mode,
                    )
                    
                    result = get_sliced_prediction(
                        original_bgr,
                        sahi_model,
                        slice_height=640,
                        slice_width=640,
                        overlap_height_ratio=0.2,
                        overlap_width_ratio=0.2,
                        verbose=False
                    )
                    
                    for obj in result.object_prediction_list:
                        if obj.mask is not None:
                            mask_polygon = obj.mask.bool_mask
                            idx = obj.category.id
                            
                            # Standard morphological post-processing for SAHI masks
                            clean_polys = post_processor.process(mask_polygon)
                            
                            if idx not in proc_polygons_by_class:
                                proc_polygons_by_class[idx] = []
                                raw_polygons_by_class[idx] = []
                                
                            proc_polygons_by_class[idx].extend(clean_polys)
                            
                            contours, _ = cv2.findContours((mask_polygon > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                            for cnt in contours:
                                raw_polygons_by_class[idx].append(cnt.reshape(-1, 2))
                else:
                    yolo_model = YOLO(cfg['path'])
                    
                    # RAW Prediction Data
                    res_raw = yolo_model.predict(original_bgr, conf=conf_slider.value, verbose=False)[0]
                    
                    if res_raw.masks is not None:
                        # Compute base display polygons
                        mask_xys = res_raw.masks.xy
                        class_ids = res_raw.boxes.cls.cpu().numpy()
                        for mask_xy, cls_id in zip(mask_xys, class_ids):
                            idx = int(cls_id)
                            if len(mask_xy) > 2:
                                if idx not in raw_polygons_by_class:
                                    raw_polygons_by_class[idx] = []
                                raw_polygons_by_class[idx].append(mask_xy.astype(np.int32))

                        # Advanced Post Processing (Watershed) based clean rendering
                        predictions = post_processor.run_pipeline(res_raw)
                        
                        for pred in predictions:
                            cls_id = int(pred["class_id"])
                            if cls_id not in proc_polygons_by_class:
                                proc_polygons_by_class[cls_id] = []
                            proc_polygons_by_class[cls_id].append(pred["polygon"])
                
                # Render the advanced comparison
                raw_hollow = visualizer.draw_hollow(original_bgr, raw_polygons_by_class)
                proc_hollow = visualizer.draw_hollow(original_bgr, proc_polygons_by_class)
                raw_solid = visualizer.draw_mask_only(original_bgr.shape, raw_polygons_by_class)
                proc_solid = visualizer.draw_mask_only(original_bgr.shape, proc_polygons_by_class)
                
                fig = visualizer.plot_advanced_comparison(
                    original_bgr, raw_hollow, proc_hollow, raw_solid, proc_solid, 
                    title_suffix=f"({cfg['name']})"
                )
                
                import datetime
                timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                save_path = f"comparison_{cfg['name']}_{timestamp}.png"
                fig.savefig(save_path, dpi=150, bbox_inches='tight', pad_inches=0)
                display(fig)
                plt.close(fig)
                
            except Exception as e:
                print(f"❌ Render failed for {cfg['name']}: {e}")
                import traceback
                traceback.print_exc()
        
        print(f"✅ Comparison generation complete!")

visualize_btn.on_click(run_multi_visualizer)
display(widgets.HTML("<b>Upload a sample to compare across configured models:</b>"))

ui_row1 = widgets.HBox([upload_compare_btn, conf_slider], layout=widgets.Layout(margin='0 0 5px 0'))
ui_row2 = widgets.HBox([min_area_slider, solidity_slider], layout=widgets.Layout(margin='0 0 10px 0'))
ui_container = widgets.VBox([ui_row1, ui_row2, visualize_btn], layout=widgets.Layout(margin='0px', padding='0px'))

display(ui_container)
display(compare_output)

HTML(value='<b>Upload a sample to compare across configured models:</b>')

Output()